# Heat Sink Baseline — FEMM Tutorial #7

Reproduction of the FEMM steady-state heat flow tutorial. Fixes the BC
bug (bottom segments insulated, not convection) and adds FEMM temperature
contour bitmap capture.

**Prerequisites:** py2femm server running on `localhost:8082`

In [ ]:
import sys
from pathlib import Path

repo_root = Path.cwd().resolve()
while repo_root.name and not (repo_root / "pyproject.toml").exists():
    repo_root = repo_root.parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

example_dir = str(repo_root / "examples" / "heatflow" / "heatsink")
if example_dir not in sys.path:
    sys.path.insert(0, example_dir)

import heatsink_tutorial as ht
import matplotlib
%matplotlib inline

assert ht.server_is_healthy(), "py2femm server not running on localhost:8082"
print("Server OK")

## 1. Geometry

In [ ]:
ht.print_dimensions()
nodes = ht.build_outline_nodes()
geo, lines = ht.build_geometry(nodes)
print(f"\n{len(nodes)} nodes, {len(lines)} segments")
ht.plot_geometry(nodes, lines)

## 2. Run FEMM (5-fin baseline)

In [ ]:
problem = ht.build_femm_problem(nodes, geo)
lua_script = ht.get_lua_script(problem)
print(f"Lua: {len(lua_script)} chars, {lua_script.count(chr(10))} lines")

csv_data = ht.run_femm(lua_script)
print(f"\nRaw output:\n{csv_data.strip()}")

## 3. Results

In [ ]:
results = ht.parse_results(csv_data)
avg_T, R_th = ht.validate_results(results)

print(f"Average temperature:  {avg_T:.1f} K  ({avg_T - 273.15:.1f} C)")
print(f"Thermal resistance:   {R_th:.2f} K/W")
print(f"Expected:             ~339 K,  R_th ~ 4.1 K/W  (full cross-section, h=10 W/m²K)")

for k, v in results.items():
    if k != "AverageTemperature_K":
        print(f"  {k} = {v:.1f} K" if isinstance(v, float) else f"  {k} = {v}")

## 4. FEMM Temperature Contour

In [ ]:
import matplotlib.pyplot as plt
import glob

# The bitmap is saved by FEMM in the job's working directory.
# Adjust this path to your server's workspace location.
WORKSPACE = r"C:\femm_workspace"  # or /mnt/c/femm_workspace for WSL

bmps = sorted(
    glob.glob(f"{WORKSPACE}/**/heatsink_temperature.bmp", recursive=True),
    key=lambda p: Path(p).stat().st_mtime,
    reverse=True,
)

if bmps:
    img = ht.load_femm_bitmap(bmps[0])
    fig, ax = plt.subplots(figsize=(10, 7))
    ax.imshow(img)
    ax.set_title("FEMM Temperature Contour")
    ax.axis("off")
    plt.tight_layout()
    plt.show()
else:
    print(f"No bitmap found in {WORKSPACE}. Check workspace path.")

## 5. Summary

In [ ]:
ht.plot_results(nodes, avg_T, R_th)